In [ ]:
import os
import sys
import subprocess
import random
import json
import requests

# 1. Mount Drive
from google.colab import drive, userdata
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. Setup Repo (Idempotent)
REPO_DIR = "/content/teachmewrite"
REPO_URL = "https://github.com/edmedpublic-hub/teachmewrite.git"

if os.path.exists(REPO_DIR):
    print("Repo exists. Pulling latest changes...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], capture_output=True)
else:
    print("Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

# 3. Install dependencies
print("Installing dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests"], check=True)

# 4. Load Secrets
try:
    os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')
    os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')
    os.environ["GCP_PROJECT"] = userdata.get('GCP_PROJECT')
    print("Secrets loaded successfully.")
except Exception as e:
    print(f"ERROR loading secrets: {e}")
    raise

# 5. Set seeds
random.seed(42)

# 6. Define Live URL
LIVE_URL = "https://teachmewrite-ai-195673983724.asia-south1.run.app"

# 7. Assertions
assert os.path.exists(f"{REPO_DIR}/ai-service/main.py"), "main.py not found!"
assert os.environ.get("ANTHROPIC_API_KEY") is not None, "ANTHROPIC_API_KEY is missing!"
print(f"Target Live Service: {LIVE_URL}")
print("✅ Cell 1a complete. Environment is ready.")

Mounted at /content/drive
Cloning repo...
Installing dependencies...
Secrets loaded successfully.
Target Live Service: https://teachmewrite-ai-195673983724.asia-south1.run.app
✅ Cell 1a complete. Environment is ready.


In [ ]:
judge_prompt = """You are a Senior English Teacher and AI Evaluator for the Pakistani education system.
You will be given a STUDENT TEXT and the AI FEEDBACK generated by an automated system.

Your job is to grade the AI FEEDBACK on three criteria using a scale of 1 to 5:
1. ACCURACY: Did the AI correctly identify the actual grammar/clarity errors in the student's text? (1 = completely wrong, 5 = perfectly accurate)
2. HELPFULNESS: Is the explanation actionable for a Pakistani student learning English? (1 = vague or confusing, 5 = clear, specific, and educational)
3. TONE: Is the feedback professional, respectful, and encouraging? (1 = rude or robotic, 5 = supportive and professional)

You must return your evaluation as a JSON object with exactly these keys:
{
  "accuracy_score": int,
  "helpfulness_score": int,
  "tone_score": int,
  "reasoning": "A brief 1-2 sentence explanation of your scores."
}

Here is the input you will evaluate:

STUDENT TEXT:
{student_text}

AI FEEDBACK:
{ai_feedback}
"""

print("Judge prompt defined successfully.")
assert len(judge_prompt) > 100, "Judge prompt is too short."
print("✅ Cell 1b complete. Judge is ready.")

Judge prompt defined successfully.
✅ Cell 1b complete. Judge is ready.


In [ ]:
# Define a tricky student text
student_text = "I am agree with my father that education is more better than money. He give me good advices every day."

print(f"Sending text to {LIVE_URL}/feedback ...")
payload = {"student_text": student_text, "level": 5}

response = requests.post(f"{LIVE_URL}/feedback", json=payload, timeout=30)

print(f"Status Code: {response.status_code}")
ai_feedback = response.json()
print(json.dumps(ai_feedback, indent=2))

assert response.status_code == 200, "API request failed"
print("✅ Cell 1c complete. Baseline captured.")

Sending text to https://teachmewrite-ai-195673983724.asia-south1.run.app/feedback ...
Status Code: 200
{
  "overall_score": 45,
  "issues": [
    {
      "category": "grammar",
      "severity": "high",
      "explanation": "Incorrect verb form: 'am agree' is not standard English. The correct form is 'agree' without 'am'.",
      "example": "I am agree"
    },
    {
      "category": "grammar",
      "severity": "high",
      "explanation": "Double comparative construction: 'more better' uses two comparative forms together. Should be either 'better' or 'more good', with 'better' being correct.",
      "example": "more better"
    },
    {
      "category": "grammar",
      "severity": "high",
      "explanation": "Subject-verb disagreement: 'He give' should be 'He gives' (third person singular present tense requires -s ending).",
      "example": "He give"
    },
    {
      "category": "grammar",
      "severity": "high",
      "explanation": "Uncountable noun error: 'advices' is inco

In [ ]:
judge_prompt = """You are a Senior English Teacher and AI Evaluator for the Pakistani education system.
You will be given a STUDENT TEXT and the AI FEEDBACK generated by an automated system.

Your job is to grade the AI FEEDBACK on three criteria using a scale of 1 to 5:
1. ACCURACY: Did the AI correctly identify the actual grammar/clarity errors in the student's text? (1 = completely wrong, 5 = perfectly accurate)
2. HELPFULNESS: Is the explanation actionable for a Pakistani student learning English? (1 = vague or confusing, 5 = clear, specific, and educational)
3. TONE: Is the feedback professional, respectful, and encouraging? (1 = rude or robotic, 5 = supportive and professional)

You must return your evaluation as a JSON object with exactly these keys:
{{
  "accuracy_score": int,
  "helpfulness_score": int,
  "tone_score": int,
  "reasoning": "A brief 1-2 sentence explanation of your scores."
}}

Here is the input you will evaluate:

STUDENT TEXT:
{student_text}

AI FEEDBACK:
{ai_feedback}
"""

print("Judge prompt redefined with escaped braces.")

Judge prompt redefined with escaped braces.


In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic"], check=True)
print("✅ Anthropic SDK installed.")

from anthropic import Anthropic

# 1. Initialize the client
judge_client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

# 2. Format the judge prompt
formatted_judge_prompt = judge_prompt.format(
    student_text=student_text,
    ai_feedback=json.dumps(ai_feedback, indent=2)
)

print("Sending to Judge...")

# 3. Call the Judge (using Haiku for speed and cost)
response = judge_client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=500,
    system="You are a strict but fair English teacher evaluator. Respond only with a valid JSON object.",
    messages=[{"role": "user", "content": formatted_judge_prompt}]
)

# 4. Parse the JSON response
judge_output = response.content[0].text
print("\n--- Judge Raw Output ---")
print(judge_output)
# Strip markdown code fences if present
judge_output = judge_output.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()

try:
    judge_json = json.loads(judge_output)
    print("\n--- Judge Scores ---")
    print(f"Accuracy: {judge_json['accuracy_score']}/5")
    print(f"Helpfulness: {judge_json['helpfulness_score']}/5")
    print(f"Tone: {judge_json['tone_score']}/5")
    print(f"Reasoning: {judge_json['reasoning']}")

    assert all(1 <= judge_json[k] <= 5 for k in ["accuracy_score", "helpfulness_score", "tone_score"]), "Scores out of bounds"

except json.JSONDecodeError:
    print("ERROR: Judge did not return valid JSON.")
    raise

print("✅ Cell 1d complete. Judge evaluation successful.")

✅ Anthropic SDK installed.
Sending to Judge...

--- Judge Raw Output ---
```json
{
  "accuracy_score": 5,
  "helpfulness_score": 5,
  "tone_score": 5,
  "reasoning": "The AI correctly identified all four major grammar errors (verb form, double comparative, subject-verb agreement, and uncountable noun) with precise explanations and a valid rewrite suggestion. The feedback is exceptionally clear, well-structured, and appropriately pitched for Pakistani students, with specific rules and examples that aid understanding."
}
```

--- Judge Scores ---
Accuracy: 5/5
Helpfulness: 5/5
Tone: 5/5
Reasoning: The AI correctly identified all four major grammar errors (verb form, double comparative, subject-verb agreement, and uncountable noun) with precise explanations and a valid rewrite suggestion. The feedback is exceptionally clear, well-structured, and appropriately pitched for Pakistani students, with specific rules and examples that aid understanding.
✅ Cell 1d complete. Judge evaluation succe

In [ ]:
judge_artifact = {
    "test_id": "baseline_001",
    "student_text": student_text,
    "ai_feedback": ai_feedback,
    "judge_eval": judge_json,
    "timestamp": "2026-10-21"  # Update to actual date
}

save_path = "/content/drive/MyDrive/teachmewrite-ai/judge_eval_day87.json"
with open(save_path, "w") as f:
    json.dump(judge_artifact, f, indent=2)

print(f"Saved judge evaluation to {save_path}")
assert os.path.exists(save_path), "Save failed!"
print("✅ Day 87 artifact saved.")

Saved judge evaluation to /content/drive/MyDrive/teachmewrite-ai/judge_eval_day87.json
✅ Day 87 artifact saved.
